In [1]:
import pandas as pd
import numpy as np

## VAN y TIR de la amortización recomendada

In [ ]:
# Datos de los préstamos (monto prestado e inter)
df = pd.read_csv("data/df_limpio.csv")
df=df[["ID", "Monto_Inicial", "Ratio_Interes"]]

# Cuadros de amortización (francés y alemán)
df_amort = pd.read_csv("data/cuadros_amort.csv")

# Sistema recomendado para cada préstamo (hay IDs repetidos, nos quedamos con el primero)
df2 = pd.read_csv("data/resultado_recomendaciones.csv", sep=";")
df = df.merge(df2[["ID", "Recomendado"]].drop_duplicates("ID"), on="ID", how="left")

# Número de meses de cada préstamo, sacado del cuadro de amortización
df = df.merge(df_amort.groupby("ID").agg(n=("Mes", "max")), on="ID", how="left")

df.head()

,ID,Monto_Inicial,Ratio_Interes,Recomendado,n
0,6B4H6E,10000,8.96,Aleman,48
1,VKTUXP,16474,14.07,Aleman,36
2,5P9OVL,40770,12.66,Frances,12
3,YL0W8G,36707,8.96,Frances,24
4,YP8WTV,86981,10.07,Frances,48


### FORMULAS - DESGLOSE
- **A** (desembolso inicial) = `Monto_Inicial`, lo que el banco va a prestar
- **Qt** (flujos de caja) = las cuotas de la amortización según sistema recomendado (fr o alem).
- **Momento de cada flujo**: igual que en el valor actual y final, las cuotas son prepagables, así que la cuota del mes k se cobra en t = k − 1.
- **k** = rentabilidad mínima que exige el banco. No viene en los datos, así que la dejamos como parámetro para poder cambiarla.

VAN = −A + Σ Qt / (1+k)^t  (aceptar si VAN > 0)  ·  0 = −A + Σ Qt / (1+TIR)^t  (aceptar si TIR > k)

In [3]:
# Rentabilidad mínima exigida (anual). CÁMBIALA por la que os den en el enunciado
k_anual = 0.05
k_mensual = (1 + k_anual) ** (1 / 12) - 1   # flujos mensuales

Montamos una tabla con una fila por préstamo y una columna por mes, con la cuota del sistema recomendado en cada mes (los meses que sobran se quedan a 0, no afectan al cálculo).

In [4]:
filas = pd.Series(np.arange(len(df)), index=df["ID"])
es_aleman_cuadro = (df_amort["ID"].map(df.set_index("ID")["Recomendado"]).astype("string").str.lower().str.contains("alem")).to_numpy()

fila = df_amort["ID"].map(filas).to_numpy()
mes = df_amort["Mes"].to_numpy()

Q = np.zeros((len(df), mes.max()))

Q[fila, mes - 1] = np.where(es_aleman_cuadro, df_amort["Al_Cuota"],df_amort["Fr_Cuota"])

A = df["Monto_Inicial"].to_numpy()
t = np.arange(mes.max())   # cuotas prepagables: la del mes k se cobra en t = k - 1

**VAN** de cada préstamo: llevamos cada cuota a hoy con k y restamos lo prestado.

In [5]:
df["VAN"] = Q @ (1 + k_mensual) ** (-t) - A

**TIR** de cada préstamo: no hay fórmula cerrada, así que la buscamos por aproximaciones sucesivas (bisección), que es lo mismo que probar tasas hasta que el VAN sale 0. Se hace para todos los préstamos a la vez. Después la pasamos a anual para poder compararla con k.

In [6]:
CF = Q.copy()
CF[:, 0] -= A

bajo = [0] * len(df)
alto = [5] * len(df)

for _ in range(60):
    medio = [(bajo[i] + alto[i]) / 2 for i in range(len(df))]

    VAN = (CF * (1 + np.array(medio)[:, None]) ** (-t)).sum(axis=1)

    for i in range(len(df)):
        if VAN[i] > 0:
            bajo[i] = medio[i]
        else:
            alto[i] = medio[i]

df["TIR_mensual"] = [(bajo[i] + alto[i]) / 2 for i in range(len(df))]
df["TIR_anual"] = (1 + df["TIR_mensual"]) ** 12 - 1

**Decisión:** se acepta el préstamo si el VAN es positivo y la TIR supera a k (con flujos simples los dos criterios coinciden).

In [7]:
df["Decision"] = np.where((df["VAN"] > 0) & (df["TIR_anual"] > k_anual), "Aceptar", "Rechazar")

van_tir = df[["ID", "Recomendado", "Monto_Inicial", "n", "VAN", "TIR_anual", "Decision"]]
van_tir.head()

,ID,Recomendado,Monto_Inicial,n,VAN,TIR_anual,Decision
0,6B4H6E,Aleman,10000,48,821.452125,0.098064,Aceptar
1,VKTUXP,Aleman,16474,36,2292.195447,0.160780,Aceptar
2,5P9OVL,Frances,40770,12,1889.237755,0.161162,Aceptar
3,YL0W8G,Frances,36707,24,1701.149692,0.102182,Aceptar
4,YP8WTV,Frances,86981,48,9541.370306,0.110563,Aceptar


In [8]:
# Resumen por sistema recomendado
resumen = df.groupby("Recomendado").agg(
    prestamos=("ID", "count"),
    VAN_medio=("VAN", "mean"),
    TIR_anual_media=("TIR_anual", "mean"),
    pct_aceptar=("Decision", lambda s: (s == "Aceptar").mean() * 100)
)
resumen

,prestamos,VAN_medio,TIR_anual_media,pct_aceptar
Recomendado,,,,
Aleman,20849,3230.536501,0.143445,100.000000
Frances,81046,4634.091993,0.116135,93.369198


In [9]:
# VAN total de la cartera
print("VAN total de la cartera:", round(df["VAN"].sum(), 2))
print("Préstamos a aceptar:", (df["Decision"] == "Aceptar").sum(), "de", len(df))


VAN total de la cartera: 442928075.15
Préstamos a aceptar: 96521 de 101895


In [10]:
#comprobación de que no hay nulos
van_tir.isna().sum()

ID               0
Recomendado      0
Monto_Inicial    0
n                0
VAN              0
TIR_anual        0
Decision         0
dtype: int64

Repetimos el VAN con varias k para ver cuánto cambia la decisión:

In [ ]:
for k_var in [0.03, 0.05, 0.08, 0.10, 0.12]:
    k_m = (1 + k_var) ** (1 / 12) - 1
    van_var = Q @ (1 + k_m) ** (-t) - A
    print(f"k = {k_var:.0%}  ->  VAN > 0 en el {(van_var > 0).mean():.1%} de los préstamos")

k = 3%  ->  VAN > 0 en el 99.0% de los préstamos
k = 5%  ->  VAN > 0 en el 94.7% de los préstamos
k = 8%  ->  VAN > 0 en el 80.8% de los préstamos
k = 10%  ->  VAN > 0 en el 66.3% de los préstamos
k = 12%  ->  VAN > 0 en el 49.8% de los préstamos


## Conclusión: viabilidad de la inversión

Con una rentabilidad mínima exigida del 5% anual, la cartera es claramente viable: se aceptan 96.521 de los 101.895 préstamos (el 94,7%), el VAN medio es de unos 4.347 € por préstamo y el VAN conjunto de la cartera supera los 440 millones de euros. En todos los préstamos el VAN y la TIR dan la misma decisión, algo esperable porque los flujos son simples (sale el préstamo una vez y después solo entran cuotas, así que hay un único cambio de signo y una única TIR).

Los 5.374 préstamos que se rechazan son los de tipo de interés más bajo (entre el 2% y el 4,7%): su TIR queda por debajo del 5% que exige el banco, o sea, rentan menos de lo que cuesta el capital y destruirían valor. Por sistemas, el alemán sale aceptado en el 100% de los casos (TIR media del 14,3%) y el francés en el 93,4% (TIR media del 11,6%).

La conclusión depende de k: si el banco exigiera un 8% se aceptaría el 80,8% de los préstamos, con un 10% el 66,3% y con un 12% menos de la mitad (49,8%). Por eso la decisión final debe hacerse con el coste de capital real del banco. Y como límite del análisis, este cálculo da por hecho que todas las cuotas se cobran; en los préstamos con más riesgo de impago el VAN real sería menor.